Create the master tables for each fund combining the accounts, projects, balance, and other tables. 

The tables will be saved in a folder in the silver layer. 

In [0]:
from pyspark.sql.functions import col, to_date

# Create Silver schema to save the silver layer data
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.silver_fund_data")

# Get list of all funds from bronze tables
bronze_tables = spark.sql("SHOW TABLES IN workspace.bronze_fund_data").toPandas()
# parse out fund names from bronze tables to a list
fund_names = sorted(set(name.split("_")[0] for name in bronze_tables["tableName"]))


for fund in fund_names:
    print(f"processing {fund}")

    try:
        
        balance = spark.table(f"workspace.bronze_fund_data.{fund}_balances")
        balance = balance.withColumn("Effective_Date", to_date(col("Effective_Date"), "yyyyMMdd"))
        balance = balance.withColumn("Amount", col("Amount").cast("double"))
        
        print(f"{fund} balance table has {balance.count()} records")
        
        # Start joining tables together...
        master = balance            #set balance the master table

        # Join accounts table
        try:
            accounts = spark.table(f"workspace.bronze_fund_data.{fund}_accounts")
            print(f"{fund} accounts table has {accounts.count()} records")
            accounts_cols = [col for col in accounts.columns if col not in balance.columns or col in ['Account_Code', 'Entity_Code']]
            master = master.join(accounts.select(*accounts_cols), on=['Account_Code', 'Entity_Code'], how="left")
            print(f"Done joining accounts table for {fund}")

        except:
            print(f"{fund} accounts table not found")


        # Join departments table - parent department descriptions are available so pull them in
        try:
            departments = spark.table(f"workspace.bronze_fund_data.{fund}_departments")
            print(f"{fund} departments table has {departments.count()} records")
            
            # Get the parent department descriptions
            dept_parent = departments.select(col("Entity_Code"), col("Department_Code"), col("Department_Description").alias("Parent_Dept_Description")) 
            departments_w_parent = departments.join(dept_parent, on =
                                            (departments.Entity_Code == dept_parent.Entity_Code) &
                                            (departments.Parent_Department_Code  == dept_parent.Department_Code),
                                            how="left").drop(dept_parent.Entity_Code, dept_parent.Department_Code)
            
            # Now join with master
            departments_cols = [col for col in departments_w_parent.columns if col not in balance.columns or col in ['Department_Code', 'Entity_Code']]
            master = master.join(departments_w_parent.select(*departments_cols), on=['Department_Code', 'Entity_Code'], how="left")
            print(f"Done joining departments table for {fund}")

        except:
            print(f"{fund} departments table not found")

        
        # Join funds table. - parent fund codes are not of the same type.
        try:
            funds_data = spark.table(f"workspace.bronze_fund_data.{fund}_funds")
            print(f"{fund} funds table has {funds_data.count()} records")
            funds_cols = [col for col in funds_data.columns if col not in balance.columns or col in ['Fund_Code', 'Entity_Code']]
            master = master.join(funds_data.select(*funds_cols), on=['Fund_Code', 'Entity_Code'], how="left")
            print(f"Done joining funds table for {fund}")

        except:
            print(f"{fund} funds table not found")
        
        # Join projects table
        try:
            projects = spark.table(f"workspace.bronze_fund_data.{fund}_projects")
            print(f"{fund} projects table has {projects.count()} records")
            projects = projects.withColumn("Project_Start_Date", to_date(col("Project_Start_Date"), "yyyyMMdd")).withColumn     ("Project_End_Date", to_date(col("Project_End_Date"), "yyyyMMdd"))

            project_cols = [col for col in projects.columns if col not in balance.columns or col in ['Project_Code', 'Entity_Code']]

            master = master.join(projects.select(*project_cols), on=['Project_Code', 'Entity_Code'], how="left")
            print(f"Done joining projects table for {fund}")

        except:
            print(f"{fund} projects table not found")


        # Save master table
        master_table_name = f"workspace.silver_fund_data.{fund}_master"
        master.write.format("delta") \
              .mode("overwrite") \
              .option("overwriteSchema", "true") \
              .saveAsTable(master_table_name)

        print(f"Created {master_table_name} table")

    except Exception as e:
        print(f"Failed to access {fund}: {e}")


print("Silver layer created and uploaded")
            

        

Export all silver tables to a volume in order to download them

In [0]:
%pip install openpyxl --break-system-packages

In [0]:
from pyspark.sql.functions import col

table_list = spark.sql("SHOW TABLES IN workspace.silver_fund_data").toPandas()
# display(table_list)

all_value = set()
for table in table_list['tableName']:
    print(table)
    df = spark.table(f"workspace.silver_fund_data.{table}")
    unique_values = df.select('Parent_Account_Code').distinct().collect()
    all_value.update(row[0] for row in unique_values)
    # print(unique_values)
    print(f"Number of unique values: {len(unique_values)}")

print(f"Total number of unique values: {len(all_value)}")
  

Need to add the following code details found in UC data dictionary in the master tables. 
- Entity Code
- Function Code
- Parent account Code
- Function Code

Create the function to join entity_codes, function_codes, and program_codes

In [0]:
from pyspark.sql.functions import col

def join_code_desc(silver_df, code_table_name):
  # Define fully qualified table names
  
  code_full_path = f"workspace.code_descriptions.{code_table_name}"

  # Read the tables to check columns
  
  code_df = spark.table(code_full_path)

  join_column, add_columns = code_df.columns[0], code_df.columns[1:]

  if join_column in silver_df.columns:
      
      try: 
        # Perform join (will have both join columns)
        result_df = silver_df.alias("stn").join(
        code_df.alias("cdt"),
        on=col(f"stn.{join_column}") == col(f"cdt.{join_column}"),
        how="left").select(
          *[col(f"stn.{c}") for c in silver_df.columns],  # all silver cols with explicit alias
          *[col(f"cdt.{c}") for c in add_columns])

        print(f"Joined {code_table_name} tables")
        print(f"Result row counts: {result_df.count()}, column counts: {len(result_df.columns)}")
        # display(result_df)
        return result_df

      except Exception as e:
          print(f"Failed to join: {e}")
  else:
      print(f"Join column '{join_column}' not found in {table_name} table")

Create the loop to run the function to create and save the updated tables

In [0]:

source_schema = "workspace.silver_fund_data"

# Create a new schema to save the enhanced tables with code descriptions
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.silver_fund_data_enhanced")

tables = spark.catalog.listTables(source_schema)

for table in tables:
   table_name = table.name
   print("processing table: ", table_name)

   silver_full_path = f"{source_schema}.{table_name}"

# first drop the extra source_file column from each table

   spark.sql(f"""ALTER TABLE {silver_full_path} \
             SET TBLPROPERTIES ('delta.columnMapping.mode' = 'name')""")
   
             
   col_to_drop = "source_file"
   spark.sql(f"""ALTER TABLE {silver_full_path} \
             DROP COLUMN IF EXISTS {col_to_drop}""")

   
   # Add parent account code description since the join
   parent_acct_add = spark.sql(f"""
                                 SELECT 
                                    pca.* , 
                                    pac.Account_Code_Description AS Parent_Account_Code_Description,
                                    pac.Account_type AS Parent_Account_Code_Type
                                 FROM {silver_full_path} AS pca
                                 LEFT JOIN workspace.code_descriptions.parent_account_codes AS pac
                                 ON pca.Parent_Account_Code = pac.Parent_Account_Code
                                 """)

   entity_code_table = "entity_code_desc" 
   entity_code_add = join_code_desc(silver_df=parent_acct_add, code_table_name=entity_code_table)

   function_code_table = "function_code_desc"
   function_code_add = join_code_desc(silver_df=entity_code_add, code_table_name=function_code_table)

   program_code_table = "program_code_desc"
   program_code_add = join_code_desc(silver_df=function_code_add, code_table_name=program_code_table)

   enhanced_table_name = f"workspace.silver_fund_data_enhanced.{table_name}_enhanced"
   program_code_add.write.format("delta") \
              .mode("overwrite") \
              .option("overwriteSchema", "true") \
              .saveAsTable(enhanced_table_name)

   print("finished table: ", table_name, "="*20)
   
   # display(program_code_add)






In [0]:
df_path = "workspace.silver_fund_data_enhanced.130a_master_enhanced"
df = spark.table(df_path)

print(df.columns)

In [0]:
cols_to_keep = [ 'Amount', 'fund_id', 'Account_Description', 'Department_Description', 'Department_Type', 'Parent_Dept_Description', 'Fund_Description', 'Parent_Fund_Code', 'Budgeted', 'Budgeted_Type', 'Endowment_to_Entity_Code', 'Endowment_Restriction_Code', 'Endowment_Purpose_Code', 'Gift_Fund', 'Gift_Fund_Endowment_Income', 'Project_Description', 'Project_Type', 'Project_Start_Date', 'Project_End_Date', 'Prime_Sponsor_Code', 'Direct_Sponsor_Code', 'Principal_Investigator_Name', 'Flow_Through_Code', 'Indirect_Cost_Base_Code', 'Indirect_Cost_Rate', 'On_Campus', 'CFDA_Agency_Number', 'CFDA_Program_Number', 'Research_and_Development', 'Award_Identifier', 'Award_Description', 'Award_Type', 'Award_Active', 'Award_Federal_ARRA', 'NSF_Code', 'SPX_PROJECT_ID', 'Parent_Account_Code_Description', 'Parent_Account_Code_Type', 'Entity_Code_Desc', 'Function_Code_Desc', 'Program_Code_Desc']

print(len(cols_to_keep))